# Full Pipeline — Videos folder → `<video_name>.json` per video

One notebook, end to end, for **any number of videos**:

```
videos + yolo11x_surv_v1.pt/.zip
   └─ Stage 1  YOLO11x + ByteTrack   → intermediate/<video>_tracks.csv
   └─ Stage 2  Temporal windowing    → intermediate/<video>_windows.json
   └─ Stage 3  Qwen2.5-VL-7B (4-bit) → outputs/<video>.json     <-- final, same format as your events.json
```

**How to use:** put all videos and the weights file (`yolo11x_surv_v1.pt` *or* `yolo11x_surv_v1.zip`) in one folder, set `INPUT_DIR` in Section 1 (or leave it `None` on Kaggle to auto-detect), then **Run All**.

Design notes
- YOLO runs for *all* videos first, then is freed from GPU memory, *then* Qwen loads. The 11x model and the 7B VLM never sit on the GPU together.
- Frames are decoded with OpenCV directly, so any container OpenCV/FFmpeg can open works (mp4, avi, mov, mkv, webm, mpg, wmv, flv, ts, 3gp, ...), not just the formats Ultralytics whitelists.
- Fully resumable: finished videos are skipped and the VLM stage checkpoints after every window. A bad video is logged and skipped; it does not kill the batch.

## 0. Install

In [ ]:
!pip install -q ultralytics
!pip install -q -U transformers accelerate bitsandbytes qwen-vl-utils

In [ ]:
import os, re, gc, json, math, time, zipfile, shutil
from pathlib import Path
from itertools import combinations
from collections import Counter

import cv2
import numpy as np
import pandas as pd
import torch
import yaml
from PIL import Image
from tqdm.auto import tqdm

print("CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only (VLM will be very slow)")

## 1. Config

Everything tunable lives here. The values are the ones from your Stage 1a/1b/2/4 notebooks.

In [ ]:
# ---------------- Paths ----------------
# Folder holding your videos AND yolo11x_surv_v1.pt (or .zip). Searched recursively.
#   Kaggle : leave None to scan /kaggle/input, or e.g. "/kaggle/input/my-dataset"
#   Colab  : e.g. "/content/videos"      Local : e.g. "./videos"
INPUT_DIR  = None
OUTPUT_DIR = None        # None -> /kaggle/working/outputs on Kaggle, else ./outputs

WEIGHTS_STEM = "yolo11x_surv_v1"      # matches yolo11x_surv_v1.pt or yolo11x_surv_v1.zip

VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv", ".webm", ".mpg", ".mpeg", ".wmv", ".flv",
              ".m4v", ".ts", ".mts", ".m2ts", ".3gp", ".ogv", ".asf", ".mxf"}

# ---------------- Stage 1: YOLO + ByteTrack ----------------
CLASS_NAMES = {0: "Person", 1: "Car", 2: "Carrying_object", 3: "Bike", 4: "Gun", 5: "Knife"}
DETECT_CONF = 0.10      # deliberately LOW so ByteTrack's 2nd association pass has boxes (Stage 1b note)
IMGSZ       = 640
BYTETRACK_CFG = {       # == bytetrack_custom.yaml
    "tracker_type": "bytetrack", "track_high_thresh": 0.5, "track_low_thresh": 0.2,
    "new_track_thresh": 0.6, "track_buffer": 30, "match_thresh": 0.8, "fuse_score": True,
}

# ---------------- Stage 2: windowing ----------------
WINDOW_SEC = 10.0
STRIDE_SEC = 5.0
STATIONARY_SPEED_PH_PER_SEC = 0.15
LOITER_DWELL_SEC            = 8.0
LOITER_MAX_DISPLACEMENT_PH  = 1.5
PROXIMITY_THRESHOLD_PH      = 1.2
HELD_PROXIMITY_SCALE        = 1.0

# ---------------- Stage 3: VLM ----------------
MODEL_ID          = "Qwen/Qwen2.5-VL-7B-Instruct"
FRAMES_PER_WINDOW = 5
MAX_FRAME_SIDE    = 896     # frames are downscaled to this longest side -> bounds VRAM/time on 1080p+ video
MAX_RETRIES       = 2
MAX_NEW_TOKENS    = 1024

# ---------------- Run behaviour ----------------
RESUME = True               # skip finished videos / reuse VLM checkpoints

# ---------------- Resolve folders ----------------
ON_KAGGLE = Path("/kaggle/working").exists()
SEARCH_ROOT = Path(INPUT_DIR) if INPUT_DIR else (Path("/kaggle/input") if ON_KAGGLE else Path("."))
OUT_DIR   = Path(OUTPUT_DIR) if OUTPUT_DIR else (Path("/kaggle/working/outputs") if ON_KAGGLE else Path("outputs"))
WORK_DIR  = OUT_DIR.parent / "pipeline_work"
INTER_DIR = WORK_DIR / "intermediate"
for d in (OUT_DIR, INTER_DIR): d.mkdir(parents=True, exist_ok=True)
print("Search root:", SEARCH_ROOT.resolve(), "\nOutputs    :", OUT_DIR.resolve())

## 2. Discover videos and weights

Videos are found recursively. If the weights are a `.zip`, it is unpacked and the `.pt` inside is used. Each video gets a unique `video_name` (its file stem; if two files share a stem, the extension / parent folder is appended) which becomes both its `video_id` and its output file name.

In [ ]:
def find_videos(root: Path):
    vids = [p for p in root.rglob("*")
            if p.is_file() and p.suffix.lower() in VIDEO_EXTS and not p.name.startswith(".")
            and OUT_DIR.resolve() not in p.resolve().parents]
    return sorted(vids)

def safe(s): return re.sub(r"[^\w.\-]+", "_", s).strip("_") or "video"

def assign_names(videos):
    stems = Counter(v.stem for v in videos)
    names, used = {}, set()
    for v in videos:
        n = safe(v.stem if stems[v.stem] == 1 else f"{v.stem}_{v.suffix.lstrip('.').lower()}")
        base, k = n, 2
        while n in used:                     # same stem+ext in different subfolders
            n = f"{base}_{safe(v.parent.name)}" if k == 2 else f"{base}_{k}"; k += 1
        used.add(n); names[v] = n
    return names

def find_weights(root: Path) -> Path:
    pts, zips, anypt = [], [], []
    for p in root.rglob("*"):
        if not p.is_file(): continue
        if p.name == WEIGHTS_STEM + ".pt":    pts.append(p)
        elif p.name == WEIGHTS_STEM + ".zip": zips.append(p)
        elif p.suffix == ".pt":               anypt.append(p)
    if pts: return pts[0]
    if zips:
        dest = WORK_DIR / "weights"; dest.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zips[0]) as z: z.extractall(dest)
        inner = sorted(dest.rglob("*.pt"), key=lambda p: (p.stem != WEIGHTS_STEM, len(str(p))))
        if not inner: raise FileNotFoundError(f"No .pt inside {zips[0]}")
        return inner[0]
    if len(anypt) == 1:
        print(f"WARNING: {WEIGHTS_STEM}.pt not found; using the only .pt present: {anypt[0].name}")
        return anypt[0]
    raise FileNotFoundError(f"Could not find {WEIGHTS_STEM}.pt/.zip under {root}")

VIDEOS = find_videos(SEARCH_ROOT)
VIDEO_NAME = assign_names(VIDEOS)
WEIGHTS = find_weights(SEARCH_ROOT)

assert VIDEOS, f"No video files found under {SEARCH_ROOT} (looked for {sorted(VIDEO_EXTS)})"
print(f"Weights: {WEIGHTS}  ({WEIGHTS.stat().st_size/1e6:.0f} MB)\n")
print(f"{len(VIDEOS)} video(s):")
for v in VIDEOS: print(f"  {v.name:<45} -> {VIDEO_NAME[v]}.json")

## 3. Stage 1 — YOLO + ByteTrack

Same logic as `SurveillanceTracker`, but frames are read with OpenCV and fed to `model.track(frame, persist=True)`. That makes it independent of Ultralytics' video-extension whitelist (e.g. `.flv`, `.3gp`, `.mts` fail there but open fine in OpenCV). The tracker state is reset between videos so IDs never bleed across footage.

In [ ]:
BYTETRACK_YAML = str(WORK_DIR / "bytetrack_custom.yaml")
with open(BYTETRACK_YAML, "w") as f: yaml.safe_dump(BYTETRACK_CFG, f, sort_keys=False)

SCHEMA_COLUMNS = ["video_id", "camera_id", "frame_id", "timestamp_sec", "x1", "y1", "x2", "y2",
                  "confidence", "class_id", "class_name", "track_id"]

def video_meta(path: Path):
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened(): raise IOError(f"OpenCV cannot open {path.name}")
    fps = cap.get(cv2.CAP_PROP_FPS)
    if not fps or fps != fps or fps <= 0 or fps > 240: fps = 25.0
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); cap.release()
    return fps, w, h, max(n, 0)

def track_video(model, path: Path, video_id: str):
    fps, W, H, n_total = video_meta(path)
    model.predictor = None                       # fresh tracker state for THIS video
    cap = cv2.VideoCapture(str(path))
    rows, idx = [], 0
    pbar = tqdm(total=n_total or None, desc=f"track {video_id}", leave=False)
    while True:
        ok, frame = cap.read()
        if not ok: break
        res = model.track(frame, tracker=BYTETRACK_YAML, conf=DETECT_CONF, imgsz=IMGSZ,
                          persist=True, verbose=False, half=torch.cuda.is_available())[0]
        if res.boxes is not None and res.boxes.id is not None:
            b = res.boxes
            xyxy = b.xyxy.cpu().numpy(); conf = b.conf.cpu().numpy()
            cls = b.cls.cpu().numpy().astype(int); tid = b.id.cpu().numpy().astype(int)
            ts = round(idx / fps, 3)
            for k in range(len(xyxy)):
                rows.append((video_id, video_id, idx, ts,
                             round(float(xyxy[k][0]), 1), round(float(xyxy[k][1]), 1),
                             round(float(xyxy[k][2]), 1), round(float(xyxy[k][3]), 1),
                             round(float(conf[k]), 3), int(cls[k]),
                             CLASS_NAMES.get(int(cls[k]), str(int(cls[k]))), int(tid[k])))
        idx += 1; pbar.update(1)
    cap.release(); pbar.close()
    if idx == 0: raise IOError(f"No frames decoded from {path.name}")
    return pd.DataFrame(rows, columns=SCHEMA_COLUMNS), dict(fps=fps, width=W, height=H, frames=idx)

## 4. Stage 2 — Temporal windowing

Ported unchanged from your Stage 2 notebook (global-bounds enter/exit fix, dedup pass, held/place/pickup association, numpy → native cast), with one improvement: zone grid uses the **real frame size** read from the video instead of the largest box seen.

In [ ]:
GRID_NAMES = ["top_left", "top_mid", "top_right", "mid_left", "center", "mid_right",
              "bottom_left", "bottom_mid", "bottom_right"]

def grid_zone(cx, cy, fw, fh):
    col = min(max(int(cx / fw * 3), 0), 2); row = min(max(int(cy / fh * 3), 0), 2)
    return GRID_NAMES[row * 3 + col]

def to_native(o):
    if isinstance(o, dict):  return {k: to_native(v) for k, v in o.items()}
    if isinstance(o, list):  return [to_native(v) for v in o]
    if isinstance(o, np.integer):  return int(o)
    if isinstance(o, np.floating): return float(o)
    if isinstance(o, np.bool_):    return bool(o)
    return o

def prep_tracks(df):
    df = df.copy()
    df["cx"] = (df.x1 + df.x2) / 2.0; df["cy"] = (df.y1 + df.y2) / 2.0; df["bbox_h"] = df.y2 - df.y1
    return df.sort_values(["video_id", "track_id", "timestamp_sec"]).reset_index(drop=True)

def person_height(df):
    p = df[df.class_name == "Person"]
    return float(p.bbox_h.median()) if len(p) else float(df.bbox_h.median())

def track_global_stats(df):
    return df.groupby(["video_id", "track_id"]).agg(
        class_name=("class_name", "first"), t_start=("timestamp_sec", "min"),
        t_end=("timestamp_sec", "max")).reset_index()

def sliding_windows(t_min, t_max, window_sec, stride_sec):
    if t_max <= t_min:
        yield (t_min, t_min + window_sec); return
    start = t_min
    while start < t_max:
        yield (round(start, 3), round(start + window_sec, 3)); start += stride_sec

def compute_window_features(win_df, global_stats, video_id, w_start, w_end, ph, fw, fh):
    tracks_out = []
    gstats = global_stats[global_stats.video_id == video_id].set_index("track_id")
    centroids = {}
    for track_id, tg in win_df.groupby("track_id"):
        tg = tg.sort_values("timestamp_sec")
        cls_name = tg.class_name.iloc[0]
        dwell = float(tg.timestamp_sec.max() - tg.timestamp_sec.min())
        step = np.sqrt(tg.cx.diff().fillna(0.0) ** 2 + tg.cy.diff().fillna(0.0) ** 2)
        path_len = float(step.sum())
        disp = float(math.hypot(tg.cx.iloc[-1] - tg.cx.iloc[0], tg.cy.iloc[-1] - tg.cy.iloc[0]))
        speed = path_len / dwell if dwell > 0 else 0.0
        zones = sorted({grid_zone(r.cx, r.cy, fw, fh) for r in tg.itertuples()})

        cands = []
        if track_id in gstats.index:
            g = gstats.loc[track_id]
            if w_start <= g.t_start < w_end: cands.append("enter")
            if w_start <= g.t_end < w_end:   cands.append("exit")
        if dwell >= LOITER_DWELL_SEC and disp / ph <= LOITER_MAX_DISPLACEMENT_PH: cands.append("loiter")
        if speed / ph <= STATIONARY_SPEED_PH_PER_SEC: cands.append("stationary")

        centroids[track_id] = list(zip(tg.timestamp_sec, tg.cx, tg.cy))
        tracks_out.append({"track_id": int(track_id), "class": cls_name, "dwell_sec": round(dwell, 2),
                           "displacement_px": round(disp, 1), "mean_speed": round(speed, 1),
                           "zones": zones, "near": [], "candidate_events": cands})

    by_id = {t["track_id"]: t for t in tracks_out}
    for a, b in combinations(centroids.keys(), 2):
        pa = {round(t, 1): (x, y) for t, x, y in centroids[a]}
        pb = {round(t, 1): (x, y) for t, x, y in centroids[b]}
        shared = sorted(set(pa) & set(pb))
        if not shared: continue
        overlap, near = 0.0, False
        for t in shared:
            if math.hypot(pa[t][0] - pb[t][0], pa[t][1] - pb[t][1]) / ph <= PROXIMITY_THRESHOLD_PH:
                near = True; overlap += 0.1
        if near:
            by_id[a]["near"].append({"track_id": int(b), "class": by_id[b]["class"], "overlap_sec": round(overlap, 1)})
            by_id[b]["near"].append({"track_id": int(a), "class": by_id[a]["class"], "overlap_sec": round(overlap, 1)})
    return tracks_out

def associate_carrying_objects(win_df, tracks_out, ph):
    by_id = {t["track_id"]: t for t in tracks_out}
    persons = win_df[win_df.class_name == "Person"]; objects = win_df[win_df.class_name == "Carrying_object"]
    if objects.empty or persons.empty: return
    thr = HELD_PROXIMITY_SCALE * ph
    for obj_id, og in objects.groupby("track_id"):
        og = og.sort_values("timestamp_sec")
        held_by, released_at, holding = None, None, None
        for row in og.itertuples():
            same_t = persons[np.isclose(persons.timestamp_sec, row.timestamp_sec, atol=0.05)]
            nearest, nd = None, float("inf")
            for p in same_t.itertuples():
                d = math.hypot(row.cx - p.cx, row.cy - p.cy)
                if d < nd: nearest, nd = p.track_id, d
            if nearest is not None and nd <= thr:
                if held_by is None: held_by = int(nearest)
                holding = True
            else:
                if holding: released_at = float(row.timestamp_sec)
                holding = False
        if obj_id in by_id:
            if held_by is not None: by_id[obj_id]["held_by"] = held_by
            if released_at is not None:
                by_id[obj_id]["released_at"] = round(released_at, 2)
                if by_id[obj_id].get("mean_speed", 999) / ph <= STATIONARY_SPEED_PH_PER_SEC:
                    by_id[obj_id]["candidate_events"].append("place")
            elif held_by is not None:
                by_id[obj_id]["candidate_events"].append("pickup")

def dedup_enter_exit(windows, global_stats):
    seen = set(); gs = global_stats.set_index(["video_id", "track_id"])
    for w in windows:
        vid = w["video_id"]
        for t in w["tracks"]:
            if "enter" in t["candidate_events"] or "exit" in t["candidate_events"]:
                new = []
                for ev in t["candidate_events"]:
                    if ev not in ("enter", "exit"): new.append(ev); continue
                    ts = gs.loc[(vid, t["track_id"])].t_start if ev == "enter" else gs.loc[(vid, t["track_id"])].t_end
                    key = (vid, t["track_id"], ev, round(float(ts), 1))
                    if key not in seen: seen.add(key); new.append(ev)
                t["candidate_events"] = new
    return windows

def build_windows(df, frame_w, frame_h):
    # df = tracks of ONE video
    df = prep_tracks(df)
    gstats = track_global_stats(df); ph = person_height(df)
    video_id = df.video_id.iloc[0]; camera_id = df.camera_id.iloc[0]
    t_min, t_max = float(df.timestamp_sec.min()), float(df.timestamp_sec.max())
    out = []
    for ws, we in sliding_windows(t_min, t_max, WINDOW_SEC, STRIDE_SEC):
        wdf = df[(df.timestamp_sec >= ws) & (df.timestamp_sec < we)]
        if wdf.empty: continue
        tr = compute_window_features(wdf, gstats, video_id, ws, we, ph, frame_w, frame_h)
        associate_carrying_objects(wdf, tr, ph)
        out.append({"video_id": video_id, "camera_id": camera_id, "window": [round(ws, 2), round(we, 2)], "tracks": tr})
    return to_native(dedup_enter_exit(out, gstats))

## 5. Run Stage 1 + 2 over every video

Writes `intermediate/<video>_tracks.csv` and `intermediate/<video>_windows.json`. Videos that already have both files are reused (`RESUME`). The YOLO model is deleted from GPU memory when this section ends.

In [ ]:
from ultralytics import YOLO

STATUS = {}     # video_name -> dict(status, ...)
yolo = None
t0 = time.time()
for v in VIDEOS:
    name = VIDEO_NAME[v]
    tr_csv, win_json = INTER_DIR / f"{name}_tracks.csv", INTER_DIR / f"{name}_windows.json"
    final = OUT_DIR / f"{name}.json"
    try:
        if RESUME and final.exists():
            STATUS[name] = {"status": "already_done"}; print(f"[skip] {name}: final JSON exists"); continue
        if RESUME and tr_csv.exists() and win_json.exists():
            STATUS[name] = {"status": "stage12_cached"}; print(f"[cache] {name}: tracks+windows reused"); continue
        if yolo is None: yolo = YOLO(str(WEIGHTS))
        df, meta = track_video(yolo, v, name)
        df.to_csv(tr_csv, index=False)
        windows = build_windows(df, meta["width"], meta["height"]) if len(df) else []
        with open(win_json, "w") as f: json.dump({"windows": windows}, f, indent=2)
        STATUS[name] = {"status": "stage12_done", **meta, "track_rows": len(df),
                        "unique_tracks": int(df.track_id.nunique()) if len(df) else 0, "windows": len(windows)}
        print(f"[ok] {name}: {len(df)} rows, {STATUS[name]['unique_tracks']} tracks, {len(windows)} windows")
    except Exception as e:
        STATUS[name] = {"status": "failed_stage12", "error": f"{type(e).__name__}: {e}"}
        print(f"[FAIL] {name}: {e}")

del yolo; gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
print(f"\nStage 1+2 finished in {time.time()-t0:.0f}s; YOLO released from memory.")

## 6. Stage 3 — Qwen2.5-VL event extraction (functions)

Same prompt shape and hallucination guard as your Stage 4 notebook (every `subject_track`/`object_track` must exist in that window, else retry, then drop). Changes, all small:

- **Retries are informative.** On a rejected answer the error is fed back to the model and sampling is switched on; plain re-asking with greedy decoding just returns the same wrong answer.
- **Event times:** the prompt now says `start`/`end` are seconds *relative to the window start*, and each event also gets `abs_start` / `abs_end` (video time). Your earlier `events.json` mixed the two conventions.
- Frames are downscaled to `MAX_FRAME_SIDE`, JSON is extracted even if the model adds text around it, and the compute dtype is fp16 on T4 (no native bf16) / bf16 where supported.

In [ ]:
SYSTEM_PROMPT = (
    "You describe video surveillance events. Reply ONLY with JSON matching the given schema, "
    "no preamble, no markdown fences. "
    "Use subject_track and object_track values from the 'Tracked objects' list below. "
    "If a candidate event is valid, confirm it. If no specific action occurs, create an event "
    "with event_type 'scene_description' and describe what the tracked objects are doing. "
    "Always return at least one descriptive event. "
    "'start' and 'end' are seconds measured from the START of the window (0 to window length)."
)
SCHEMA_HINT = ('{"events": [{"event_type": str, "subject_track": int_or_null, "object_track": int_or_null, '
               '"start": float, "end": float, "zone": str, "confidence": float, "description": str}]}')

def build_user_prompt(window):
    ws, we = window["window"]
    lines = [f"Tracked objects in window {ws}-{we}s (window length {round(we-ws,2)}s):"]
    cands = []
    for t in window["tracks"]:
        d = f"- {t['class']} #{t['track_id']}: present {t['dwell_sec']}s"
        if t.get("zones"):              d += f", zones: {' -> '.join(t['zones'])}"
        if t.get("held_by") is not None:     d += f", held by #{t['held_by']}"
        if t.get("released_at") is not None: d += f", released at {t['released_at']}s"
        lines.append(d)
        for ev in t.get("candidate_events", []): cands.append(f"{ev}({t['track_id']})")
    lines += ["", f"Candidate events from motion rules: [{', '.join(cands)}]",
              "Describe the actions in these frames. Schema: " + SCHEMA_HINT]
    return "\n".join(lines)

def extract_frames(path, t_start, t_end, n):
    cap = cv2.VideoCapture(str(path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    if fps <= 0 or fps != fps or fps > 240: fps = 25.0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frames = []
    for t in np.linspace(t_start, max(t_end, t_start + 0.1), n):
        idx = int(t * fps)
        if total > 0: idx = min(idx, total - 1)
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ok, bgr = cap.read()
        if not ok:                                    # some containers seek better by time
            cap.set(cv2.CAP_PROP_POS_MSEC, t * 1000.0); ok, bgr = cap.read()
        if not ok: continue
        img = Image.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
        if max(img.size) > MAX_FRAME_SIDE:
            s = MAX_FRAME_SIDE / max(img.size)
            img = img.resize((max(28, int(img.width * s)), max(28, int(img.height * s))), Image.BICUBIC)
        frames.append(img)
    cap.release()
    return frames

VLM = {"model": None, "processor": None}

def load_vlm():
    from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
    cuda = torch.cuda.is_available()
    dt = torch.bfloat16 if (cuda and torch.cuda.is_bf16_supported()) else torch.float16
    kw = dict(quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=dt,
                                                     bnb_4bit_quant_type="nf4")) if cuda else {}
    VLM["model"] = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL_ID, device_map={"": 0} if cuda else "auto", torch_dtype=dt, **kw)
    VLM["processor"] = AutoProcessor.from_pretrained(MODEL_ID)
    print(f"Loaded {MODEL_ID}" + (f" | GPU mem {torch.cuda.memory_allocated()/1e9:.1f} GB" if cuda else " on CPU"))

def call_qwen(frames, user_prompt, sample=False):
    from qwen_vl_utils import process_vision_info
    model, proc = VLM["model"], VLM["processor"]
    messages = [{"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": [{"type": "image", "image": f} for f in frames]
                                            + [{"type": "text", "text": user_prompt}]}]
    text = proc.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    imgs, vids = process_vision_info(messages)
    inputs = proc(text=[text], images=imgs, videos=vids, padding=True, return_tensors="pt").to(model.device)
    gen_kw = dict(do_sample=True, temperature=0.3, top_p=0.9) if sample else dict(do_sample=False)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, **gen_kw)
    trimmed = [o[len(i):] for i, o in zip(inputs.input_ids, out)]
    return proc.batch_decode(trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]

def _load_json_lenient(raw):
    cleaned = re.sub(r"^```(json)?|```$", "", raw.strip(), flags=re.MULTILINE).strip()
    try: return json.loads(cleaned)
    except json.JSONDecodeError:
        a, b = cleaned.find("{"), cleaned.rfind("}")
        if a != -1 and b > a: return json.loads(cleaned[a:b + 1])
        raise

def _as_id(x):
    if x is None: return None
    if isinstance(x, str): x = x.strip().lstrip("#")
    return int(x)

def parse_and_validate(raw, window):
    parsed = _load_json_lenient(raw)
    if isinstance(parsed, list): parsed = {"events": parsed}
    if "events" not in parsed or not isinstance(parsed["events"], list):
        raise ValueError("Response missing top-level 'events' list")
    known = {t["track_id"] for t in window["tracks"]}
    ws, we = window["window"]; wl = we - ws
    out = []
    for ev in parsed["events"]:
        subj, obj = _as_id(ev.get("subject_track")), _as_id(ev.get("object_track"))
        if subj is not None and subj not in known: raise ValueError(f"subject_track {subj} not in window (known: {sorted(known)})")
        if obj is not None and obj not in known:   raise ValueError(f"object_track {obj} not in window (known: {sorted(known)})")
        s = float(ev.get("start", 0.0)); e = float(ev.get("end", wl))
        zone = ev.get("zone")
        out.append({"event_type": ev.get("event_type", "scene_description"),
                    "subject_track": subj, "object_track": obj,
                    "start": s, "end": e,
                    "abs_start": round(ws + min(max(s, 0.0), wl), 2),
                    "abs_end":   round(ws + min(max(e, 0.0), wl), 2),
                    "zone": zone if isinstance(zone, str) or zone is None else str(zone),
                    "confidence": float(ev.get("confidence", 0.8)),
                    "description": str(ev.get("description", ""))})
    return out

def process_window(window, video_path):
    frames = extract_frames(video_path, window["window"][0], window["window"][1], FRAMES_PER_WINDOW)
    if not frames:
        print(f"  window {window['window']}: no frames extracted, skipping"); return []
    base = build_user_prompt(window); prompt, err = base, None
    for attempt in range(MAX_RETRIES + 1):
        raw = call_qwen(frames, prompt, sample=attempt > 0)
        try: return parse_and_validate(raw, window)
        except (ValueError, TypeError, json.JSONDecodeError) as e:
            err = e; print(f"  window {window['window']}: attempt {attempt+1} rejected -- {e}")
            prompt = (base + f"\n\nYour previous answer was rejected: {e}. Valid track ids: "
                      f"{sorted(t['track_id'] for t in window['tracks'])}. Return corrected JSON only.")
    print(f"  window {window['window']}: dropped after {MAX_RETRIES+1} attempts (last error: {err})")
    return []

## 7. Run Stage 3 over every video → `outputs/<video_name>.json`

Output format is identical to your current `events.json`: `{"windows": [{video_id, camera_id, window, tracks, events}, ...]}`. A per-video checkpoint (`<video>.partial.json`) is written after every window, so an interrupted run resumes mid-video.

In [ ]:
def run_vlm_video(name, video_path):
    win_json = INTER_DIR / f"{name}_windows.json"
    partial  = INTER_DIR / f"{name}.partial.json"
    windows = json.load(open(win_json))["windows"]
    done = {}
    if RESUME and partial.exists():
        done = {tuple(w["window"]): w.get("events", []) for w in json.load(open(partial))["windows"]}
        if done: print(f"  resuming: {len(done)}/{len(windows)} windows already done")
    n_drop = 0
    for i, w in enumerate(tqdm(windows, desc=f"VLM {name}", leave=False)):
        key = tuple(w["window"])
        if key in done: w["events"] = done[key]; continue
        w["events"] = process_window(w, video_path)
        if not w["events"]: n_drop += 1
        with open(partial, "w") as f: json.dump({"windows": windows[:i + 1]}, f)
    with open(OUT_DIR / f"{name}.json", "w") as f: json.dump({"windows": windows}, f, indent=2)
    if partial.exists(): partial.unlink()
    return len(windows), sum(len(w["events"]) for w in windows), n_drop

todo = [v for v in VIDEOS if STATUS.get(VIDEO_NAME[v], {}).get("status") in ("stage12_done", "stage12_cached")]
print(f"{len(todo)} video(s) need the VLM stage.")
if todo:
    load_vlm()
    t0 = time.time()
    for v in todo:
        name = VIDEO_NAME[v]; print(f"\n=== {name} ===")
        try:
            nw, ne, nd = run_vlm_video(name, v)
            STATUS[name].update(status="done", vlm_windows=nw, events=ne, windows_without_events=nd)
            print(f"[ok] {name}: {nw} windows, {ne} events, {nd} windows dropped -> {OUT_DIR/(name+'.json')}")
        except Exception as e:
            STATUS[name].update(status="failed_stage3", error=f"{type(e).__name__}: {e}")
            print(f"[FAIL] {name}: {e}")
    print(f"\nStage 3 finished in {(time.time()-t0)/60:.1f} min")

## 8. Summary and download

In [ ]:
with open(OUT_DIR / "run_summary.json", "w") as f: json.dump(STATUS, f, indent=2)

print(f"{'video':<40}{'status':<18}{'windows':>8}{'events':>8}")
for n, s in STATUS.items():
    print(f"{n:<40}{s['status']:<18}{s.get('vlm_windows', s.get('windows', '')):>8}{s.get('events', ''):>8}"
          + (f"   {s['error']}" if 'error' in s else ""))

zip_path = shutil.make_archive(str(OUT_DIR.parent / "pipeline_outputs"), "zip", OUT_DIR)
print("\nFinal JSONs:", sorted(p.name for p in OUT_DIR.glob("*.json") if p.name != "run_summary.json"))
print("Zipped:", zip_path)
try:
    from IPython.display import FileLink, display
    display(FileLink(zip_path))
except Exception: pass

## Notes

- **Re-running after an error:** just Run All again; finished work is reused. Set `RESUME = False` to force a full redo (also delete `pipeline_work/` if you changed the weights or Stage 2 thresholds, since cached tracks/windows are reused otherwise).
- **Windows with no events** (`windows_without_events`) are ones where the VLM never produced valid, non-hallucinated JSON after retries. They keep their `tracks` data and carry `"events": []`, as in your original design.
- **Time/GPU:** YOLO11x runs on every frame, so cost scales with total video length; the VLM cost scales with `(duration / STRIDE_SEC)` windows. If a T4 is slow, lower `FRAMES_PER_WINDOW` to 3 first.